# Task 1: Data Handling & Memory Management

The raw TIM Big Data Challenge release for Milan is **~20 GB** of tab-separated text
(one file per day, 62 days). Each raw row is
`(square_id, time_interval, country_code, sms_in, sms_out, call_in, call_out, internet_traffic)`,
and the assignment only needs **(square_id, time_interval, internet_traffic)** with
`internet_traffic` summed across `country_code`.

Loading 20 GB into a single DataFrame is not viable on a workstation, so the pipeline:

1. Reads each raw `.txt` in row-chunks (`pd.read_csv(chunksize=...)`).
2. Keeps only the 3 required columns at read-time (`usecols`).
3. Downcasts dtypes (`square_id`→int16, `internet_traffic`→float32).
4. Aggregates per chunk: `groupby(square_id, time_interval).sum()`.
5. Streams each aggregated file straight to a Snappy-compressed Parquet shard.

Downstream Tasks 2 and 3 then read **only the shards / areas they need** via pyarrow
predicate pushdown — they never materialise the full dataset.

In [ ]:
import sys, os, time, platform
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import pyarrow as pa

from src.data.loader import (
    RAW_COLUMNS, KEEP_COLUMNS, OPTIMIZED_DTYPES,
    build_parquet_dataset, load_area_from_parquet, total_traffic_per_area,
    memory_usage_mb,
)

RAW_DIR     = '../data/raw/'
PARQUET_DIR = '../data/processed/milan_traffic_parquet/'
os.makedirs(PARQUET_DIR, exist_ok=True)

## 1.1 Hardware & Software Setup

In [ ]:
import torch
print('Platform   :', platform.platform())
print('Processor  :', platform.processor())
print('Python     :', platform.python_version())
print('pandas     :', pd.__version__)
print('numpy      :', np.__version__)
print('pyarrow    :', pa.__version__)
print('torch      :', torch.__version__)
print('CUDA       :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU       :', torch.cuda.get_device_name(0))

raw_files = sorted(f for f in os.listdir(RAW_DIR) if f.endswith('.txt'))
raw_total = sum(os.path.getsize(os.path.join(RAW_DIR, f)) for f in raw_files) / (1024**3)
print(f'\nRaw files found: {len(raw_files)}  ({raw_total:.2f} GB total)')

## 1.2 Baseline: naïve load of a single file

Read **one** day with default `pd.read_csv` settings (all 8 columns, default dtypes) to
establish a baseline memory footprint.

In [ ]:
sample_file = os.path.join(RAW_DIR, raw_files[0])
t0 = time.perf_counter()
df_baseline = pd.read_csv(sample_file, sep='\t', header=None, names=RAW_COLUMNS)
baseline_load_s = time.perf_counter() - t0

mem_baseline = memory_usage_mb(df_baseline)
print(f'Baseline (1 day, 8 cols, default dtypes):')
print(f'  rows         : {len(df_baseline):,}')
print(f'  memory       : {mem_baseline:.1f} MB')
print(f'  load time    : {baseline_load_s:.1f} s')
print(f'\nDtypes:\n{df_baseline.dtypes}')

size_on_disk = os.path.getsize(sample_file) / (1024**2)
print(f'\nFile on disk : {size_on_disk:.1f} MB')
print(f'Expansion    : {mem_baseline / size_on_disk:.1f}x (in-RAM vs on-disk)')

# Extrapolate naively to the full dataset
full_naive_gb = mem_baseline * len(raw_files) / 1024
print(f'\nNaïve full-dataset estimate: {full_naive_gb:.1f} GB in RAM '
      '— infeasible on this workstation.')

## 1.3 Optimised single-file load

Apply the four optimisations and re-measure on the same file:

| Optimisation | Effect |
|---|---|
| `usecols=[square_id, time_interval, internet_traffic]` | drops 5 columns at read time |
| `dtype` downcast (int16 / float32) | halves bytes-per-cell |
| `groupby(square_id, time_interval).sum()` | collapses ~200× duplication from `country_code` |
| Parquet + Snappy on disk | 5–10× smaller than .txt; fast random reads |

In [ ]:
t0 = time.perf_counter()
df_one = pd.read_csv(
    sample_file, sep='\t', header=None,
    names=RAW_COLUMNS, usecols=KEEP_COLUMNS,
    dtype={'square_id': 'int32', 'time_interval': 'int64',
           'internet_traffic': 'float64'},
)
df_one['internet_traffic'] = df_one['internet_traffic'].fillna(0).astype('float32')
df_one['square_id']        = df_one['square_id'].astype('int16')
df_one_agg = (df_one.groupby(['square_id', 'time_interval'], sort=False)
                    ['internet_traffic'].sum().reset_index())
df_one_agg['internet_traffic'] = df_one_agg['internet_traffic'].astype('float32')
optimised_load_s = time.perf_counter() - t0

print(f'Optimised single-file load:')
print(f'  rows after groupby : {len(df_one_agg):,} '
      f'(was {len(df_baseline):,} — '
      f'{len(df_baseline)/max(len(df_one_agg),1):.1f}× duplication from country_code)')
print(f'  memory             : {memory_usage_mb(df_one_agg):.2f} MB')
print(f'  load+aggregate     : {optimised_load_s:.1f} s')
print(f'\nReduction vs baseline: '
      f'{mem_baseline/max(memory_usage_mb(df_one_agg),1e-6):.1f}x smaller in RAM')

## 1.4 Stream the full dataset to a partitioned Parquet store

`build_parquet_dataset` does the same per-file pipeline above for every raw `.txt`,
but **chunks each file** (`chunksize=1_000_000` rows) so peak RAM never exceeds a few
hundred MB regardless of file size. Each day becomes one Parquet shard on disk.

The job is idempotent — re-running it skips shards that already exist.

In [ ]:
summary = build_parquet_dataset(
    raw_dir=RAW_DIR,
    out_dir=PARQUET_DIR,
    chunksize=1_000_000,
)
summary.head(10)

In [ ]:
raw_gb     = raw_total
parquet_gb = summary['shard_mb'].sum() / 1024
print(f'Raw on disk        : {raw_gb:.2f} GB')
print(f'Parquet on disk    : {parquet_gb:.2f} GB '
      f'({raw_gb/max(parquet_gb,1e-6):.1f}x smaller)')
print(f'Total aggregate s  : {summary["wall_seconds"].sum():.1f}')
if summary['raw_rows'].notna().any():
    print(f'Raw rows processed : {int(summary["raw_rows"].sum()):,}')
    print(f'Aggregated rows    : {int(summary["aggregated_rows"].sum()):,}')

## 1.5 Verify: selective read of a single area

The whole point of the partitioned layout is that downstream notebooks pull only
the areas they need. pyarrow pushes the `square_id` predicate down to the file
scan, so this read costs O(rows_for_one_area) RAM, not O(dataset).

In [ ]:
t0 = time.perf_counter()
df_4159 = load_area_from_parquet(PARQUET_DIR, 4159)
print(f'Loaded area 4159 in {time.perf_counter()-t0:.2f}s — '
      f'{len(df_4159):,} rows, {memory_usage_mb(df_4159):.2f} MB in RAM')
print(df_4159.head())

## 1.6 Streaming aggregate: total traffic per area (no full load)

`total_traffic_per_area` walks the Parquet dataset one row-group at a time and
accumulates a per-area sum via `np.bincount`. Used by Task 2 to (i) pick the
highest-traffic area and (ii) build the PDF over all 10,000 areas.

In [ ]:
t0 = time.perf_counter()
totals = total_traffic_per_area(PARQUET_DIR)
print(f'Computed totals for {len(totals):,} areas in '
      f'{time.perf_counter()-t0:.1f}s — peak RAM ≈ one row group.')
print(f'Top 5 areas by total traffic:')
print(totals.sort_values(ascending=False).head())

# Persist for the EDA notebook so we don't have to recompute.
totals.to_frame().to_parquet('../data/processed/total_per_area.parquet')

## 1.7 Challenges & limitations

- **Memory ceiling.** A naïve full-dataset load would need ~`{full_naive_gb}` GB of RAM
  (extrapolated from one day). The streaming + Parquet design above keeps peak RAM
  bounded by `chunksize × few columns`, ~a few hundred MB.
- **Country-code duplication.** Each `(square_id, time_interval)` appears once *per
  foreign country prefix* in the raw data. Without the per-chunk `groupby().sum()` the
  output would (a) be ~200× larger than necessary and (b) be semantically wrong as a
  univariate time series.
- **NaN handling.** The raw `internet_traffic` field is NaN for rows where a country
  had no internet activity in that interval. We replace NaN→0 *before* aggregation so
  the sum is well-defined.
- **Resumability.** Each shard is written atomically; restarting the job picks up
  where it left off rather than re-processing from scratch.
- **Disk vs RAM trade.** Parquet+Snappy is ~5–10× smaller than the raw `.txt`, and
  random reads on a single area cost milliseconds.

Hardware used for this run is printed in §1.1 above.